In [2]:
# lets load our libraries
#!conda install -y -c conda-forge numpy=1.19.5

from rdkit import Chem
from rdkit.Chem import Draw

import os
import sys
import rdkit
import h5py

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri 
import rdkit.Chem
import rdkit.Chem.AllChem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import mpl_toolkits.mplot3d
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from collections import Counter


# topology stuff 
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from gtda.diagrams import PersistenceEntropy
from gtda.diagrams import NumberOfPoints
from gtda.diagrams import Amplitude
from sklearn.pipeline import make_union, Pipeline

# fixc this at some point
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

#import projection
#from projection.molecule import Molecule
#from projection.pdbmolecule import PDBMolecule
#from projection.mol2molecule import Mol2Molecule

import tensorflow as tf
print("TensorFlow version: " + tf.__version__)

import deepchem as dc
import helper_functions as h
#from projection.face import Face

TensorFlow version: 2.7.0


In [80]:
save_dir=r'C:\Users\eg16993\OneDrive - University of Bristol\Documents\Datasets\topol_datasets'

out_file_name="PDBBind_general_2020_topological_features.hdf5"
# Open hdf5 file, calc basic details
#outfile = out_file_name
print(out_file_name)
fh = h5py.File(os.path.join(save_dir,out_file_name), 'r+')
num_of_rows, num_of_molecules=h.basic_info_hdf5_dataset(fh, label='PDB_code')
#fh.close()

PDBBind_general_2020_topological_features.hdf5
num_of_ rows is:	14109
num_of_molecules is:	 14109
num_of_augments is:	1


In [81]:
fh.keys()

<KeysViewHDF5 ['-logKd_over_Ki', 'C:\\Users\\eg16993\\Documents\\GitHub\\graphs_and_topology_for_chemistry\\SelectedPDBCodes.hdf5', 'C:\\Users\\eg16993\\OneDrive - University of Bristol\\Documents\\Datasets\\topol_datasets\\SelectedPDBCodes.hdf5', 'Kd_over_Ki', 'L_bottle_1', 'L_bottle_2', 'L_bottle_3', 'L_landsc_1', 'L_landsc_2', 'L_landsc_3', 'L_no_p_1', 'L_no_p_2', 'L_no_p_3', 'L_pers_S_1', 'L_pers_S_2', 'L_pers_S_3', 'L_pers_img_1', 'L_pers_img_2', 'L_pers_img_3', 'L_wasser_1', 'L_wasser_2', 'L_wasser_3', 'PDB_code', 'P_bottle_1', 'P_bottle_2', 'P_bottle_3', 'P_landsc_1', 'P_landsc_2', 'P_landsc_3', 'P_no_p_1', 'P_no_p_2', 'P_no_p_3', 'P_pers_S_1', 'P_pers_S_2', 'P_pers_S_3', 'P_pers_img_1', 'P_pers_img_2', 'P_pers_img_3', 'P_wasser_1', 'P_wasser_2', 'P_wasser_3', 'SelectedPDBCodes', 'SelectedPDBCodes.hdf5', 'ligand_name', 'molID', 'reference', 'release_year', 'resolution']>

In [41]:
fh['PDB_code'][:]

array([b'3zzf', b'3gww', b'1w8l', ..., b'1avd', b'2xui', b'2avi'],
      dtype=object)

In [15]:
data_dir=r'C:\Users\eg16993\OneDrive - University of Bristol\Documents\Datasets\PDBbind_2016_plain_text_index.tar\PDBbind_2016_plain_text_index\index'
#RUN THI
name_file_name="INDEX_general_PL_name.2016"
data_file_name="INDEX_general_PL_data.2016"
name_column_list_name=['PDB code', 'release year', 'Uniprot ID', 'protein name']
if not name_file_name == '':
        ## This reads in the pdb codes for each protein in the core dataset
        fh = open(os.path.join(data_dir, name_file_name), 'r')
        c = 0
        lines = []
        for line in fh.readlines():
            if not line.startswith('#'):
                if c == 0:
                    words = [x for x in line.split(' ') if not x == '']
                    #last_word = words[-1]
                    #last_word = last_word[-1].strip()
                    first_word = words[0] 
                    #last_word = '_'.join(last_word)
                    #new_line = words
                    # print(new_line)
                    lines.append(first_word)
                    # df_line = pd.DataFrame([new_line],columns=column_list)
                    # print(df_line)
                    # df_index_core.append(df_line)
                    # labels = line

        fh.close()
        #df_index_gen = pd.DataFrame(lines, columns=name_column_list_name)

In [75]:
import os
import h5py

# Define the list of PDB codes to extract
lines = ['3eql', '1zyr', '3dxj', '4zh4', '4zh3']

# Define the input and output file names and paths
in_file_name = 'PDBBind_general_2020_topological_features.hdf5'
out_file_name = 'SelectedPDBData.hdf5'
in_file_path = os.path.join(save_dir, in_file_name)
out_file_path = os.path.join(save_dir, out_file_name)

# Open the HDF5 file in read mode
with h5py.File(in_file_path, 'r') as f_in:
    # Open the new HDF5 file in write mode
    with h5py.File(out_file_path, 'w') as f_out:
        f_out.create_dataset('PDB_code', shape=(len(lines), 48))
        for code in lines:
            if code.encode('utf-8') in f_out['PDB_code']:
                # Get the index of the current PDB code in the original HDF5 file
                idx = list(f_in['PDB_code']).index(code.encode('utf-8'))
                
                # Copy all the datasets associated with this PDB code to the new file
                for key in f_in.keys():
                    f_in.copy(f_in[key], f_out, name=f'{code}/{key}')
            f_out.flush()

f_in.close()
f_out.close()
                    
                
print(f"Saved new HDF5 file to {out_file_path}")


Saved new HDF5 file to C:\Users\eg16993\OneDrive - University of Bristol\Documents\Datasets\topol_datasets\SelectedPDBData.hdf5


C:\Users\eg16993\Anaconda3\envs\tdaf-tf2p7\lib\site-packages\ipykernel_launcher.py:19: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison


In [79]:
fh = h5py.File(os.path.join('SelectedPDBCodes.hdf5'), 'r+')
fh.keys()
#num_of_rows, num_of_molecules=h.basic_info_hdf5_dataset(fh, label='molID')

<KeysViewHDF5 ['C:\\Users\\eg16993\\Documents\\GitHub\\graphs_and_topology_for_chemistry\\SelectedPDBCodes.hdf5']>

In [55]:
import os
import h5py

# Define the list of PDB codes to extract
lines = ['3eql', '1zyr', '3dxj', '4zh4', '4zh3']

# Define the input and output file names and paths
in_file_name = 'PDBBind_general_2020_topological_features.hdf5'
out_file_name = 'SelectedPDBCodes.hdf5'
in_file_path = os.path.join(save_dir, in_file_name)
out_file_path = os.path.join(os.getcwd(), out_file_name)

print(f"Saved new HDF5 file to {out_file_path}")

fh = h5py.File(os.path.join(save_dir,in_file_name), 'r')

# Open the HDF5 file in read/write mode
with h5py.File(out_file_path, 'w') as f:
    # Retrieve the PDBCode dataset from the file
    pdbcodes_dataset = fh['PDB_code']

    # Determine the subset of indices corresponding to the desired PDB codes
    selected_indices = [i for i, code in enumerate(pdbcodes_dataset) if code.decode('utf-8') in lines]

    # Create a new dataset with the desired subset of PDB codes
    selected_codes = f.create_dataset(out_file_path, 
                                      shape=(len(selected_indices),), 
                                      dtype=pdbcodes_dataset.dtype)
    selected_codes[...] = pdbcodes_dataset[selected_indices]
    
    # save changes 
    f.flush()
    
    
f.close()
fh.close()


Saved new HDF5 file to C:\Users\eg16993\Documents\GitHub\graphs_and_topology_for_chemistry\SelectedPDBCodes.hdf5


In [51]:
selected_indices

[1829, 2110, 4071, 5120, 6452]

In [47]:
print(f"Saved new HDF5 file to {out_file_path}")

Saved new HDF5 file to C:\Users\eg16993\Documents\GitHub\graphs_and_topology_for_chemistry\SelectedPDBCodes.hdf5


In [30]:
os.getcwd()

'C:\\Users\\eg16993\\Documents\\GitHub\\graphs_and_topology_for_chemistry'

In [27]:
selected_codes

<Closed HDF5 dataset>

In [59]:
fh = h5py.File(os.path.join(os.getcwd(),'SelectedPDBCodes.hdf5'), 'r+')
num_of_rows, num_of_molecules=h.basic_info_hdf5_dataset(fh, label='PDB_code')

KeyError: "Unable to open object (object 'PDB_code' doesn't exist)"

In [16]:
lines

['3eql',
 '1zyr',
 '3dxj',
 '4zh4',
 '4zh3',
 '4zh2',
 '4xsx',
 '4xsy',
 '4xsz',
 '4mex',
 '4kn4',
 '4kn7',
 '4kmu',
 '4waf',
 '4l23',
 '3hhm',
 '4ykn',
 '3zim',
 '4jps',
 '4tv3',
 '1t3t',
 '1i1e',
 '1g9d',
 '1g9a',
 '1g9b',
 '1g9c',
 '2nm1',
 '2j9n',
 '1i6v',
 '1y4z',
 '4ci1',
 '4ci2',
 '4ci3',
 '4qsh',
 '4qsk',
 '1bxr',
 '3i3d',
 '3i3b',
 '3dyo',
 '3t0d',
 '3vdb',
 '3t08',
 '1px4',
 '3muz',
 '3t2p',
 '3t2q',
 '3t09',
 '3mv0',
 '3vd9',
 '3t0b',
 '3vd7',
 '3vd4',
 '3vdc',
 '1hty',
 '1qx1',
 '1ps3',
 '2ow6',
 '2f1b',
 '2f1a',
 '1qwu',
 '3d52',
 '1hxk',
 '2fyv',
 '3dx1',
 '2f18',
 '3d51',
 '3ddf',
 '3d50',
 '2ow7',
 '3d4z',
 '3d4y',
 '3ddg',
 '3dx3',
 '2f7p',
 '3ejs',
 '3eju',
 '3dx2',
 '3dx4',
 '3ejt',
 '3dx0',
 '2f7o',
 '1hww',
 '3ejp',
 '3ejr',
 '3ejq',
 '3w5n',
 '2x09',
 '2vzr',
 '3a3y',
 '4amw',
 '4amx',
 '2x2i',
 '4ret',
 '4res',
 '3n23',
 '2jbu',
 '4re9',
 '4ifh',
 '4dtt',
 '4gsc',
 '4dwk',
 '4gs8',
 '4nxo',
 '4lte',
 '3e4a',
 '2o9j',
 '2oa0',
 '2agv',
 '4j2t',
 '3nal',
 '4ycl',
 

In [12]:

#cluster_file_name = "INDEX_refined_cluster.2016"

df_index_refined, df_data_refined, df_cluster_refined = h.read_in_PDBBind_data(
    data_dir,
    name_file_name=name_file_name,
    data_file_name=data_file_name,
    cluster_file_name = "")

ValueError: 4 columns passed, passed data had 12 columns